In [36]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

In [37]:
# Import data
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\CDC.csv', parse_dates=['obs_date'])

# Rename 'obs_date' to 'date' and set it as the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data.set_index('date', inplace=True)

# Display the first 10 rows
data.head(10)

,indicator,API_UserName,OpDiv,observations,curr_date
date,,,,,
2025-01-01,146.71.50.198,818860012482918321,CDC,1,2025-01-01
2025-01-01,149.36.49.225,818860012482918321,CDC,28,2025-01-01
2025-01-01,162.142.125.242,818860012482918321,CDC,3,2025-01-01
2025-01-01,162.142.125.247,818860012482918321,CDC,2,2025-01-01
2025-01-01,185.230.63.171,818860012482918321,CDC,6,2025-01-01
2025-01-01,23.26.221.12,818860012482918321,CDC,49,2025-01-01
2025-01-01,23.26.221.2,818860012482918321,CDC,51,2025-01-01
2025-01-01,23.26.221.4,818860012482918321,CDC,36,2025-01-01
2025-01-01,34.160.111.145,818860012482918321,CDC,4,2025-01-01


In [38]:
def create_feature_enriched_dataset(pivot_df, history_days=14, forecast_days=7):
    X, y, indicator_names = [], [], []

    for indicator in pivot_df.columns:
        series = pivot_df[indicator].fillna(0).astype(int).values
        dates = pivot_df.index

        for i in range(history_days, len(series) - forecast_days):
            window = series[i - history_days:i]
            future_window = series[i:i + forecast_days]
            label = 1 if future_window.sum() > 0 else 0

            # Feature engineering
            days_since_seen = history_days - np.argmax(window[::-1]) if np.any(window) else history_days
            rolling_3 = window[-3:].sum()
            rolling_7 = window[-7:].sum()
            avg_14 = window.mean()
            max_14 = window.max()

            features = [
                *window,             # original raw activity (14)
                days_since_seen,    # 1
                rolling_3,          # 1
                rolling_7,          # 1
                avg_14,             # 1
                max_14              # 1
            ]

            X.append(features)
            y.append(label)
            indicator_names.append(indicator)

    return np.array(X), np.array(y), np.array(indicator_names)


In [39]:
# Convert observations to binary appearance flags
pivot_df = data.pivot_table(index='date', columns='indicator', values='observations', aggfunc='count')
pivot_df = (pivot_df > 0).astype(int)

# Ensure daily frequency and fill missing values
pivot_df = pivot_df.asfreq('D').fillna(0)
top_indicators = pivot_df.sum().nlargest(50).index  # Select top 10 indicators
pivot_df = pivot_df[top_indicators]


In [40]:
X, y, indicators = create_feature_enriched_dataset(pivot_df)

print(f"X shape: {X.shape}")        # [samples, 14]
print(f"y shape: {y.shape}")        # [samples]
print(f"Sample input:\n{X[0]}")
print(f"Sample label: {y[0]} (for indicator: {indicators[0]})")


X shape: (4150, 19)
y shape: (4150,)
Sample input:
[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. 14.  0.  0.  0.
  0.]
Sample label: 0 (for indicator: 104.21.48.1)


In [41]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Predict and evaluate
y_pred = model.predict(X)
print(classification_report(y, y_pred))

baseline_metrics = classification_report(y, y_pred, output_dict=True)


              precision    recall  f1-score   support

           0       0.64      0.84      0.73      1930
           1       0.81      0.59      0.68      2220

    accuracy                           0.71      4150
   macro avg       0.73      0.72      0.71      4150
weighted avg       0.73      0.71      0.70      4150



In [42]:
# === Step 1: Split into train/test ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# === Step 2: Train Random Forest ===
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', random_state=42)

In [43]:
# === Step 3: Predict and Evaluate ===
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# === Step 4: Report ===
print(" Classification Report:")
print(classification_report(y_test, y_pred))

print(" Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print(f" ROC AUC Score: {roc_auc:.4f}")


 Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.77      0.66       386
           1       0.72      0.51      0.60       444

    accuracy                           0.63       830
   macro avg       0.65      0.64      0.63       830
weighted avg       0.65      0.63      0.63       830

 Confusion Matrix:
[[297  89]
 [217 227]]
 ROC AUC Score: 0.6766
